# Spark Session Initialization

Initialize the Spark Session used for all DataFrame operations in this notebook.

In [1]:
import os
import sys
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip



# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

builder = ( SparkSession.builder \
    .appName("BGG Data Validation") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.LocalLogStore")
            
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Paths Configuration

Define all input and output paths used in this notebook.

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
DATA_PATH = PROJECT_ROOT / "data"

BRONZE_PATH = DATA_PATH / "bronze"
SOURCE_PATH = DATA_PATH / "source"
# REFERENCE_PATH = DATA_PATH / "reference"

GAMES_BGG_CSV = SOURCE_PATH / "bgg_db_4999_shortened.csv"

## Add project root to sys.path to enable importing functions from utils
sys.path.append(str(Path().resolve().parent))

# Load Reference Data and Fetch IDs

Load a Reference Delta table and extract a list of values from an ID column.

In [4]:
# from utils.data_io import load_ids

# country_ids = load_ids(REFERENCE_PATH, spark, "geography",'country_id')
# ga_ids = load_ids(REFERENCE_PATH, spark, "google_analytics",'ga_id')
# delivery_ids = load_ids(REFERENCE_PATH, spark, "delivery",'delivery_id')
# vendor_ids = load_ids(REFERENCE_PATH, spark, "vendors",'vendor_id')

# DataFrame Schemas Definition

Define structured schemas for:
- bgg_games
- customers
- employees
- sales

In [5]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, StringType, DoubleType, TimestampType, DateType
)


games_bgg_schema = StructType([
    StructField("rank", IntegerType(), True),
    StructField("bgg_url", StringType(), False),
    StructField("game_id", IntegerType(), False),
    StructField("names", StringType(), False),
    StructField("min_players", IntegerType(), True),
    StructField("max_players", IntegerType(), True),
    StructField("avg_time", IntegerType(), True),
    StructField("min_time", IntegerType(), True),
    StructField("max_time", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("avg_rating", DoubleType(), True),
    StructField("geek_rating", DoubleType(), True),
    StructField("num_votes", IntegerType(), True),
    StructField("image_url", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("mechanic", StringType(), True),
    StructField("owned", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("designer", StringType(), True),
    StructField("weight", DoubleType(), True),
])


customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("country_id", IntegerType(), False),
    StructField("registration_date", DateType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), False),
])


employees_schema = StructType([
    StructField("employee_id", StringType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("hire_date", DateType(), True),
    StructField("birth_date", DateType(), True),
    StructField("country_id", IntegerType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), False),
])


sales_schema = StructType([
    StructField("sale_id", LongType(), False),
    StructField("sale_timestamp", TimestampType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("game_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_cost", DoubleType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("currency_code", StringType(), False),
    StructField("payment_method_code", StringType(), False),
    StructField("delivery_id", StringType(), False),
    StructField("employee_id", StringType(), False),
    StructField("vendor_id", StringType(), False),
    StructField("ga_id", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("source_system", StringType(), False),
])

# Load BoardGameGeek Games CSV

Read the shortened BGG games [dataset](https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek?select=games.csv) CSV from Kaggle into a Spark DataFrame.

In [6]:
from pyspark.sql.functions import current_timestamp, lit

games_bgg_df = (
    spark.read
    .option("header", True)
    .schema(games_bgg_schema)
    .csv(str(GAMES_BGG_CSV))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("bgg.com"))
)

# Generate Random Customers and Employees 

Create synthetic customer and employee records for bronze Delta tables using Faker python library.

In [ ]:
# from faker import Faker
# import random
# from datetime import datetime, timedelta

# SEED = 58

# fake = Faker()
# Faker.seed(58)
# random.seed(SEED)

In [ ]:
# from pathlib import Path
# from pyspark.sql import DataFrame

# def save_to_bronze(
#     df: DataFrame,
#     table_name: str,
#     path: Path = BRONZE_PATH,
#     mode: str = "overwrite") -> None:

#     output_path = path / table_name

#     (
#         df.write
#         .format("delta")
#         .mode(mode)
#         .save(str(output_path))
#     )

#     print(f"Saved to bronze layer delta table: {output_path}")

In [ ]:
# def generate_customers(
#     num_customers: int,
#     country_ids: list) -> list:

#     rows = []

#     for customer_id in range(1, num_customers + 1):

#         registration_date = fake.date_between(
#             start_date="-2y",
#             end_date="today"
#         )

#         rows.append((
#             customer_id,
#             fake.first_name(),
#             fake.last_name(),
#             fake.unique.email(),
#             random.choice(country_ids),
#             registration_date,
#             datetime.utcnow(),
#             "faker_python_library"
#         ))

#     return rows

## Customers

In [ ]:
from utils.data_io import generate_customers

customer_rows = generate_customers(num_customers=2000,country_ids=country_ids)
customers_df = spark.createDataFrame(customer_rows, schema=customers_schema)

In [ ]:
customers_df.show(5)

## Employees

In [ ]:
# def generate_employees(num_employees: int, country_ids: list) -> list:

#     rows = []

#     for emp_id in range(1, num_employees + 1):

#         first = fake.first_name()
#         last = fake.last_name()
#         email = fake.unique.email()
#         phone = fake.phone_number()
#         birth = fake.date_of_birth(minimum_age=18, maximum_age=65)
#         hire = fake.date_between(start_date="-2y", end_date="-90d")
#         country = random.choice(country_ids)

#         rows.append((
#             emp_id,
#             first,
#             last,
#             email,
#             phone,
#             hire,
#             birth,
#             country,
#             datetime.utcnow(),
#             "faker_python_library"
#         ))

#     return rows

In [ ]:
from utils.data_io import generate_employees

employees_rows = generate_employees(num_employees=12, country_ids=country_ids)
employees_df = spark.createDataFrame(employees_rows,schema=employees_schema)

In [ ]:
employees_df.show(5)

# Save Bronze Tables

Writing to `delta` bronze-level tables:
- bgg_games
- customers
- employees

In [7]:
from utils.data_io import save_to_bronze

save_to_bronze(games_bgg_df, "bgg_games")
save_to_bronze(customers_df, "customers")
save_to_bronze(employees_df, "employees")

# Generate and Save Random Sales

In [ ]:
# from utils.data_io import load_ids

# games_ids = load_ids(BRONZE_PATH, spark, "bgg_games", 'game_id')
# customer_ids = load_ids(BRONZE_PATH, spark, "customers", 'customer_id')
# employee_ids = load_ids(BRONZE_PATH, spark, "employees", 'employee_id')

In [ ]:
# def generate_sales(num_sales: int):

#     rows = []

#     for sale_id in range(1, num_sales + 1):

#         quantity = random.randint(1, 3)

#         # Cost lower than price (realistic margin)
#         unit_cost = round(random.uniform(40.00, 90.00), 2)

#         # Price strictly in required range
#         unit_price = round(random.uniform(44.99, 99.99), 2)

#         # Ensure business logic: cost must be lower than price
#         if unit_cost >= unit_price:
#             unit_cost = round(unit_price * random.uniform(0.5, 0.8), 2)

#         sale_time = fake.date_time_between(start_date="-1y", end_date="-1m")

#         rows.append((
#             sale_id,
#             sale_time,
#             random.choice(customer_ids),
#             random.choice(games_ids),
#             quantity,
#             unit_cost,
#             unit_price,
#             "EUR", 
#             random.choice(["CARD", "CASH", "TRANSFER"]),
#             random.choice(delivery_ids),
#             random.choice(employee_ids),
#             random.choice(vendor_ids),
#             random.choice(ga_ids),
#             datetime.utcnow(),
#             "faker_python_library"
#         ))

#     return rows

In [8]:
from utils.data_io import generate_sales

sales_rows = generate_sales(10000, spark)

sales_df = spark.createDataFrame(
    sales_rows,
    schema=sales_schema
)

In [10]:
save_to_bronze(sales_df, "sales")

Saved to bronze layer delta table: D:\Python\Project_Boardgames_Data_Validation\data\bronze\sales


Read from `delta` and fetch id's

In [16]:
# sales_df.show(5)

In [14]:
# df_sales = sales_df.toPandas()

In [15]:
# df_sales